# Hands-On Lab: Migrating a PySpark Pipeline to Snowflake with Cortex Code

## Overview

In this lab you will migrate a financial analytics pipeline to **Snowflake-native Snowpark Python** using the `spark-migration` skill in Cortex Code.

## What You Will Do

| Step | Where | What |
|------|-------|------|
| 1 | This notebook | Create the Snowflake environment and load source data |
| 2 | Cortex Code (local) | Run the SMA migration skill to convert PySpark code to Snowpark |
| 3 | Browser | Review the EWI dashboard and run the automatic EWI fixer |
| 4 | This notebook | Execute the converted Snowpark pipeline and validate the output |

## Lab Architecture

```
+-----------------------------------------------------------------+
|  PART A - Cortex Code (local machine)                          |
|                                                                 |
|  transaction_pipeline.py  -->  SMA CLI  -->  Snowpark output  |
|  (PySpark source)              (converts)   (converted .py)   |
+--------------------------------+--------------------------------+
                                 | EWI Dashboard (opens in browser)
+--------------------------------v--------------------------------+
|  PART B - This Snowflake Notebook                              |
|                                                                 |
|  ACCOUNTS ---+                                                  |
|              +--> Snowpark Pipeline --> MONTHLY_SPEND_SUMMARY  |
|  TRANSACTIONS+                                                  |
+-----------------------------------------------------------------+
```

## Prerequisites

| Requirement | Notes |
|-------------|-------|
| Snowflake account | `SYSADMIN` or equivalent role |
| Cortex Code | Snowflake CLI with AI assistance — used in Part A |
| SMA CLI | Snowflake Migration Accelerator binary — used in Part A |
| Test workload | `test_workload_for_demo/` folder (this notebook lives inside it) |

> **Note:** The SMA CLI is distributed separately from Cortex Code. If you do not have it, ask your Snowflake account team or SE contact.

---

## Section 1: Snowflake Environment Setup

Run the SQL cells below to create the warehouse, database, and schema used throughout this lab.

In [ ]:
-- Create a dedicated warehouse for this lab
CREATE WAREHOUSE IF NOT EXISTS LAB_WH
    WAREHOUSE_SIZE = 'X-SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    COMMENT = 'Hands-on lab: PySpark to Snowflake migration';

USE WAREHOUSE LAB_WH;

In [ ]:
-- Create the database and schema used by the pipeline
CREATE DATABASE IF NOT EXISTS DEMO_DB
    COMMENT = 'Migration lab database';

CREATE SCHEMA IF NOT EXISTS DEMO_DB.FINANCE
    COMMENT = 'Finance pipeline schema';

USE DATABASE DEMO_DB;
USE SCHEMA FINANCE;

---

## Section 2: Load Source Data

The synthetic finance dataset is embedded directly in this notebook — no external files or stage uploads are required.

- **ACCOUNTS** (50 rows): account holders with type (CHECKING / SAVINGS / CREDIT), status (ACTIVE / INACTIVE / FROZEN), and credit limit
- **TRANSACTIONS** (300 rows): DEBIT, CREDIT, and TRANSFER transactions spanning January 2023 to June 2024

These tables represent the data that the original PySpark pipeline read from Snowflake using the legacy Spark-Snowflake connector.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.types import StructType, StructField, StringType, DoubleType

session = get_active_session()

# Schema for the ACCOUNTS table
accounts_schema = StructType([
    StructField('ACCOUNT_ID',    StringType(), False),
    StructField('CUSTOMER_NAME', StringType(), True),
    StructField('ACCOUNT_TYPE',  StringType(), True),
    StructField('OPENED_DATE',   StringType(), True),
    StructField('STATUS',        StringType(), True),
    StructField('CREDIT_LIMIT',  DoubleType(), True),
])

# 50 synthetic accounts (35 ACTIVE, 10 INACTIVE, 5 FROZEN)
accounts_data = [
    ('ACC001','James Mitchell','CHECKING','2018-03-15','ACTIVE',5000.0),
    ('ACC002','Sarah Chen','SAVINGS','2019-07-22','ACTIVE',10000.0),
    ('ACC003','Michael Torres','CREDIT','2020-01-10','ACTIVE',25000.0),
    ('ACC004','Emily Johnson','CHECKING','2018-11-05','ACTIVE',3000.0),
    ('ACC005','David Park','SAVINGS','2021-04-18','ACTIVE',8000.0),
    ('ACC006','Lisa Anderson','CREDIT','2019-09-30','ACTIVE',15000.0),
    ('ACC007','Robert Kim','CHECKING','2020-06-14','ACTIVE',4500.0),
    ('ACC008','Jennifer White','SAVINGS','2018-02-28','ACTIVE',12000.0),
    ('ACC009','William Brown','CREDIT','2021-08-07','ACTIVE',30000.0),
    ('ACC010','Amanda Davis','CHECKING','2022-03-19','ACTIVE',2500.0),
    ('ACC011','Christopher Wilson','SAVINGS','2019-12-01','ACTIVE',9000.0),
    ('ACC012','Stephanie Martinez','CREDIT','2020-05-23','ACTIVE',20000.0),
    ('ACC013','Kevin Thompson','CHECKING','2018-07-16','ACTIVE',6000.0),
    ('ACC014','Rachel Garcia','SAVINGS','2021-11-09','ACTIVE',7500.0),
    ('ACC015','Brian Lee','CREDIT','2019-04-25','ACTIVE',18000.0),
    ('ACC016','Melissa Clark','CHECKING','2022-01-30','ACTIVE',3500.0),
    ('ACC017','Andrew Rodriguez','SAVINGS','2020-09-12','ACTIVE',11000.0),
    ('ACC018','Nicole Lewis','CREDIT','2018-06-20','ACTIVE',22000.0),
    ('ACC019','Daniel Hall','CHECKING','2021-02-14','ACTIVE',4000.0),
    ('ACC020','Heather Young','SAVINGS','2019-08-03','ACTIVE',6500.0),
    ('ACC021','Joshua Hernandez','CREDIT','2020-11-27','ACTIVE',16000.0),
    ('ACC022','Samantha King','CHECKING','2022-05-08','ACTIVE',2000.0),
    ('ACC023','Ryan Wright','SAVINGS','2018-10-15','ACTIVE',14000.0),
    ('ACC024','Ashley Lopez','CREDIT','2021-06-29','ACTIVE',28000.0),
    ('ACC025','Matthew Hill','CHECKING','2019-03-07','ACTIVE',5500.0),
    ('ACC026','Megan Scott','SAVINGS','2020-07-21','ACTIVE',8500.0),
    ('ACC027','Justin Green','CREDIT','2018-12-13','ACTIVE',35000.0),
    ('ACC028','Brittany Adams','CHECKING','2022-09-24','ACTIVE',3000.0),
    ('ACC029','Nicholas Baker','SAVINGS','2021-01-06','ACTIVE',10500.0),
    ('ACC030','Amber Gonzalez','CREDIT','2019-05-18','ACTIVE',19000.0),
    ('ACC031','Tyler Nelson','CHECKING','2020-03-02','ACTIVE',4500.0),
    ('ACC032','Lauren Carter','SAVINGS','2018-08-29','ACTIVE',13000.0),
    ('ACC033','Brandon Mitchell','CREDIT','2021-10-11','ACTIVE',24000.0),
    ('ACC034','Kayla Perez','CHECKING','2022-07-17','ACTIVE',2800.0),
    ('ACC035','Austin Roberts','SAVINGS','2019-01-23','ACTIVE',9500.0),
    ('ACC036','Courtney Turner','CHECKING','2020-04-08','INACTIVE',5000.0),
    ('ACC037','Jordan Phillips','SAVINGS','2018-05-14','INACTIVE',7000.0),
    ('ACC038','Taylor Campbell','CREDIT','2021-09-26','INACTIVE',15000.0),
    ('ACC039','Morgan Parker','CHECKING','2019-11-30','INACTIVE',3200.0),
    ('ACC040','Alex Evans','SAVINGS','2022-02-15','INACTIVE',8000.0),
    ('ACC041','Casey Edwards','CHECKING','2020-08-19','INACTIVE',4000.0),
    ('ACC042','Riley Collins','SAVINGS','2018-04-03','INACTIVE',6000.0),
    ('ACC043','Quinn Stewart','CREDIT','2021-07-22','INACTIVE',12000.0),
    ('ACC044','Avery Morris','CHECKING','2019-10-07','INACTIVE',2500.0),
    ('ACC045','Jordan Rogers','SAVINGS','2022-06-13','INACTIVE',9000.0),
    ('ACC046','Blake Reed','CREDIT','2020-12-28','FROZEN',50000.0),
    ('ACC047','Drew Cook','CHECKING','2018-09-16','FROZEN',4800.0),
    ('ACC048','Skyler Morgan','SAVINGS','2021-03-05','FROZEN',11000.0),
    ('ACC049','River Bailey','CREDIT','2019-07-11','FROZEN',40000.0),
    ('ACC050','Sage Cooper','CHECKING','2022-04-20','FROZEN',3600.0),
]

accounts_df = session.create_dataframe(accounts_data, schema=accounts_schema)
accounts_df.write.mode('overwrite').save_as_table('ACCOUNTS')
row_count = session.table('ACCOUNTS').count()
print(f'ACCOUNTS loaded: {row_count} rows')
session.table('ACCOUNTS').show(5)

In [ ]:
# Schema for the TRANSACTIONS table
transactions_schema = StructType([
    StructField('TRANSACTION_ID',    StringType(), False),
    StructField('ACCOUNT_ID',        StringType(), True),
    StructField('AMOUNT',            DoubleType(), True),
    StructField('TRANSACTION_TYPE',  StringType(), True),
    StructField('TRANSACTION_DATE',  StringType(), True),
    StructField('MERCHANT_CATEGORY', StringType(), True),
    StructField('DESCRIPTION',       StringType(), True),
])

# 300 synthetic transactions spanning 2023-01 to 2024-06
transactions_data = [
    ('TXN0001','ACC001',45.5,'DEBIT','2023-01-05','GROCERIES','Grocery store purchase'),
    ('TXN0002','ACC001',28.75,'DEBIT','2023-01-12','DINING','Restaurant lunch'),
    ('TXN0003','ACC001',120.0,'DEBIT','2023-01-20','UTILITIES','Electric bill payment'),
    ('TXN0004','ACC001',65.25,'DEBIT','2023-02-03','GROCERIES','Weekly groceries'),
    ('TXN0005','ACC001',15.99,'DEBIT','2023-02-14','ENTERTAINMENT','Streaming subscription'),
    ('TXN0006','ACC001',500.0,'TRANSFER','2023-02-28','RETAIL','Online shopping transfer'),
    ('TXN0007','ACC001',89.5,'DEBIT','2023-03-10','DINING','Dinner with family'),
    ('TXN0008','ACC001',200.0,'DEBIT','2023-03-22','TRAVEL','Hotel booking'),
    ('TXN0009','ACC001',55.0,'DEBIT','2023-04-05','GROCERIES','Supermarket run'),
    ('TXN0010','ACC001',1200.0,'DEBIT','2023-04-18','TRAVEL','Flight tickets'),
    ('TXN0011','ACC001',35.5,'DEBIT','2023-05-02','DINING','Coffee shop'),
    ('TXN0012','ACC001',78.25,'DEBIT','2023-05-15','HEALTHCARE','Pharmacy purchase'),
    ('TXN0013','ACC002',2500.0,'CREDIT','2023-01-07','RETAIL','Salary deposit'),
    ('TXN0014','ACC002',95.0,'DEBIT','2023-01-19','GROCERIES','Monthly groceries'),
    ('TXN0015','ACC002',450.0,'DEBIT','2023-02-05','UTILITIES','Combined utilities'),
    ('TXN0016','ACC002',180.0,'DEBIT','2023-02-20','HEALTHCARE','Doctor visit'),
    ('TXN0017','ACC002',320.0,'DEBIT','2023-03-08','RETAIL','Clothing purchase'),
    ('TXN0018','ACC002',75.5,'DEBIT','2023-03-25','DINING','Anniversary dinner'),
    ('TXN0019','ACC002',1800.0,'CREDIT','2023-04-01','RETAIL','Tax refund'),
    ('TXN0020','ACC002',42.0,'DEBIT','2023-04-12','ENTERTAINMENT','Movie tickets'),
    ('TXN0021','ACC002',110.0,'DEBIT','2023-05-07','GROCERIES','Bulk grocery shop'),
    ('TXN0022','ACC002',250.0,'TRANSFER','2023-05-22','RETAIL','Transfer to checking'),
    ('TXN0023','ACC003',3200.0,'DEBIT','2023-01-08','TRAVEL','International flight'),
    ('TXN0024','ACC003',145.75,'DEBIT','2023-01-23','DINING','Business dinner'),
    ('TXN0025','ACC003',890.0,'DEBIT','2023-02-10','RETAIL','Electronics purchase'),
    ('TXN0026','ACC003',55.25,'DEBIT','2023-02-25','GROCERIES','Convenience store'),
    ('TXN0027','ACC003',4200.0,'DEBIT','2023-03-15','TRAVEL','Hotel for conference'),
    ('TXN0028','ACC003',220.0,'DEBIT','2023-03-28','ENTERTAINMENT','Concert tickets'),
    ('TXN0029','ACC003',380.0,'CREDIT','2023-04-10','RETAIL','Expense reimbursement'),
    ('TXN0030','ACC003',95.5,'DEBIT','2023-04-22','DINING','Team lunch'),
    ('TXN0031','ACC003',175.0,'DEBIT','2023-05-08','HEALTHCARE','Medical consultation'),
    ('TXN0032','ACC003',2750.0,'DEBIT','2023-05-20','TRAVEL','Travel package'),
    ('TXN0033','ACC004',67.25,'DEBIT','2023-01-09','GROCERIES','Weekly shop'),
    ('TXN0034','ACC004',32.5,'DEBIT','2023-01-24','DINING','Lunch break'),
    ('TXN0035','ACC004',150.0,'DEBIT','2023-02-12','UTILITIES','Internet and phone'),
    ('TXN0036','ACC004',88.75,'DEBIT','2023-02-26','RETAIL','Home goods'),
    ('TXN0037','ACC004',22.99,'DEBIT','2023-03-09','ENTERTAINMENT','App subscription'),
    ('TXN0038','ACC004',420.0,'DEBIT','2023-03-23','HEALTHCARE','Dental cleaning'),
    ('TXN0039','ACC004',48.5,'DEBIT','2023-04-14','GROCERIES','Farmers market'),
    ('TXN0040','ACC004',290.0,'DEBIT','2023-04-28','RETAIL','Seasonal clothing'),
    ('TXN0041','ACC004',15.75,'DEBIT','2023-05-10','DINING','Coffee and snack'),
    ('TXN0042','ACC004',560.0,'DEBIT','2023-05-24','TRAVEL','Weekend getaway'),
    ('TXN0043','ACC005',102.5,'DEBIT','2023-01-11','GROCERIES','Monthly staples'),
    ('TXN0044','ACC005',680.0,'DEBIT','2023-01-28','UTILITIES','Annual insurance'),
    ('TXN0045','ACC005',44.25,'DEBIT','2023-02-14','DINING','Valentine dinner'),
    ('TXN0046','ACC005',1500.0,'CREDIT','2023-02-28','RETAIL','Freelance income'),
    ('TXN0047','ACC005',195.0,'DEBIT','2023-03-17','RETAIL','Sporting goods'),
    ('TXN0048','ACC005',68.0,'DEBIT','2023-03-30','HEALTHCARE','Prescription refill'),
    ('TXN0049','ACC005',75.0,'DEBIT','2023-04-16','GROCERIES','Specialty foods'),
    ('TXN0050','ACC005',340.0,'DEBIT','2023-04-29','ENTERTAINMENT','Music festival tickets'),
    ('TXN0051','ACC006',1650.0,'DEBIT','2023-01-13','TRAVEL','Business trip hotel'),
    ('TXN0052','ACC006',88.25,'DEBIT','2023-01-30','DINING','Client entertainment'),
    ('TXN0053','ACC006',425.0,'DEBIT','2023-02-16','RETAIL','Office supplies'),
    ('TXN0054','ACC006',35.0,'DEBIT','2023-03-02','GROCERIES','Quick grocery run'),
    ('TXN0055','ACC006',3500.0,'DEBIT','2023-03-19','TRAVEL','Conference package'),
    ('TXN0056','ACC006',118.75,'DEBIT','2023-04-04','HEALTHCARE','Eye exam and glasses'),
    ('TXN0057','ACC006',250.0,'CREDIT','2023-04-20','RETAIL','Travel reimbursement'),
    ('TXN0058','ACC006',72.5,'DEBIT','2023-05-06','DINING','Working lunch'),
    ('TXN0059','ACC006',845.0,'DEBIT','2023-05-21','RETAIL','Tech accessories'),
    ('TXN0060','ACC007',58.75,'DEBIT','2023-01-15','GROCERIES','Supermarket'),
    ('TXN0061','ACC007',29.99,'DEBIT','2023-01-31','ENTERTAINMENT','Streaming bundle'),
    ('TXN0062','ACC007',175.0,'DEBIT','2023-02-18','UTILITIES','Gas and electric'),
    ('TXN0063','ACC007',460.0,'DEBIT','2023-03-04','RETAIL','Shoes and apparel'),
    ('TXN0064','ACC007',39.5,'DEBIT','2023-03-21','DINING','Thai takeout'),
    ('TXN0065','ACC007',920.0,'DEBIT','2023-04-07','TRAVEL','Car rental for vacation'),
    ('TXN0066','ACC007',82.25,'DEBIT','2023-04-23','HEALTHCARE','Urgent care visit'),
    ('TXN0067','ACC007',130.0,'DEBIT','2023-05-11','GROCERIES','Bulk buy'),
    ('TXN0068','ACC007',55.0,'TRANSFER','2023-05-27','RETAIL','Peer payment'),
    ('TXN0069','ACC008',3000.0,'CREDIT','2023-01-16','RETAIL','Monthly salary'),
    ('TXN0070','ACC008',112.5,'DEBIT','2023-01-29','GROCERIES','Grocery haul'),
    ('TXN0071','ACC008',380.0,'DEBIT','2023-02-20','UTILITIES','Quarterly water bill'),
    ('TXN0072','ACC008',67.25,'DEBIT','2023-03-06','DINING','Family brunch'),
    ('TXN0073','ACC008',1100.0,'DEBIT','2023-03-24','RETAIL','Home appliance'),
    ('TXN0074','ACC008',48.0,'DEBIT','2023-04-09','ENTERTAINMENT','Theme park tickets'),
    ('TXN0075','ACC008',235.0,'DEBIT','2023-04-25','HEALTHCARE','Specialist visit'),
    ('TXN0076','ACC008',93.75,'DEBIT','2023-05-14','GROCERIES','Organic market'),
    ('TXN0077','ACC009',4800.0,'DEBIT','2023-01-17','TRAVEL','Luxury hotel stay'),
    ('TXN0078','ACC009',225.0,'DEBIT','2023-02-01','DINING','Fine dining'),
    ('TXN0079','ACC009',1350.0,'DEBIT','2023-02-22','RETAIL','Designer purchase'),
    ('TXN0080','ACC009',78.5,'DEBIT','2023-03-10','GROCERIES','Wine and gourmet'),
    ('TXN0081','ACC009',2900.0,'DEBIT','2023-03-28','TRAVEL','Spa resort weekend'),
    ('TXN0082','ACC009',165.0,'DEBIT','2023-04-14','ENTERTAINMENT','Private event'),
    ('TXN0083','ACC009',88.25,'DEBIT','2023-04-30','HEALTHCARE','Wellness program'),
    ('TXN0084','ACC009',3400.0,'DEBIT','2023-05-16','TRAVEL','International trip'),
    ('TXN0085','ACC009',445.0,'CREDIT','2023-05-31','RETAIL','Refund processed'),
    ('TXN0086','ACC010',72.5,'DEBIT','2023-01-18','GROCERIES','Weekly groceries'),
    ('TXN0087','ACC010',18.99,'DEBIT','2023-02-04','ENTERTAINMENT','Music service'),
    ('TXN0088','ACC010',295.0,'DEBIT','2023-02-24','UTILITIES','Home insurance premium'),
    ('TXN0089','ACC010',41.75,'DEBIT','2023-03-12','DINING','Quick lunch'),
    ('TXN0090','ACC010',650.0,'DEBIT','2023-03-29','RETAIL','Furniture piece'),
    ('TXN0091','ACC010',105.25,'DEBIT','2023-04-15','HEALTHCARE','Chiropractor session'),
    ('TXN0092','ACC010',58.0,'DEBIT','2023-05-01','GROCERIES','Midweek top-up'),
    ('TXN0093','ACC010',180.0,'TRANSFER','2023-05-18','RETAIL','Rent payment split'),
    ('TXN0094','ACC011',88.5,'DEBIT','2023-06-03','GROCERIES','Weekly shop'),
    ('TXN0095','ACC011',320.0,'DEBIT','2023-06-20','UTILITIES','Combined bills'),
    ('TXN0096','ACC011',54.25,'DEBIT','2023-07-08','DINING','Dinner out'),
    ('TXN0097','ACC011',750.0,'DEBIT','2023-07-25','TRAVEL','Train tickets and hotel'),
    ('TXN0098','ACC011',38.75,'DEBIT','2023-08-12','GROCERIES','Produce market'),
    ('TXN0099','ACC011',195.0,'DEBIT','2023-08-28','RETAIL','Back to school shopping'),
    ('TXN0100','ACC012',1250.0,'DEBIT','2023-06-05','RETAIL','Laptop purchase'),
    ('TXN0101','ACC012',67.5,'DEBIT','2023-06-22','DINING','Sushi dinner'),
    ('TXN0102','ACC012',2100.0,'DEBIT','2023-07-10','TRAVEL','Summer vacation package'),
    ('TXN0103','ACC012',145.0,'DEBIT','2023-07-27','HEALTHCARE','Annual physical'),
    ('TXN0104','ACC012',88.25,'DEBIT','2023-08-14','GROCERIES','Grocery run'),
    ('TXN0105','ACC012',420.0,'DEBIT','2023-08-30','ENTERTAINMENT','Concert and drinks'),
    ('TXN0106','ACC013',55.75,'DEBIT','2023-06-07','GROCERIES','Supermarket'),
    ('TXN0107','ACC013',175.0,'DEBIT','2023-06-24','UTILITIES','Electric bill'),
    ('TXN0108','ACC013',29.99,'DEBIT','2023-07-12','ENTERTAINMENT','Gaming subscription'),
    ('TXN0109','ACC013',380.0,'DEBIT','2023-07-29','TRAVEL','Road trip expenses'),
    ('TXN0110','ACC013',62.5,'DEBIT','2023-08-16','DINING','BBQ restaurant'),
    ('TXN0111','ACC013',210.0,'DEBIT','2023-08-31','RETAIL','Hardware store'),
    ('TXN0112','ACC014',1800.0,'CREDIT','2023-06-09','RETAIL','Freelance payment'),
    ('TXN0113','ACC014',98.75,'DEBIT','2023-06-26','GROCERIES','Monthly shop'),
    ('TXN0114','ACC014',45.0,'DEBIT','2023-07-14','DINING','Takeaway dinner'),
    ('TXN0115','ACC014',560.0,'DEBIT','2023-07-31','HEALTHCARE','Dental crown'),
    ('TXN0116','ACC014',82.25,'DEBIT','2023-08-18','GROCERIES','Bulk groceries'),
    ('TXN0117','ACC014',135.0,'DEBIT','2023-09-02','ENTERTAINMENT','Annual museum pass'),
    ('TXN0118','ACC015',3100.0,'DEBIT','2023-06-11','TRAVEL','Business class upgrade'),
    ('TXN0119','ACC015',188.5,'DEBIT','2023-06-28','RETAIL','Wardrobe refresh'),
    ('TXN0120','ACC015',92.75,'DEBIT','2023-07-16','DINING','Client lunch'),
    ('TXN0121','ACC015',1450.0,'DEBIT','2023-07-30','ENTERTAINMENT','VIP event package'),
    ('TXN0122','ACC015',275.0,'DEBIT','2023-08-20','HEALTHCARE','Private health test'),
    ('TXN0123','ACC015',68.0,'DEBIT','2023-09-04','GROCERIES','Convenience shop'),
    ('TXN0124','ACC016',76.25,'DEBIT','2023-06-13','GROCERIES','Grocery store'),
    ('TXN0125','ACC016',220.0,'DEBIT','2023-06-30','UTILITIES','Quarterly gas bill'),
    ('TXN0126','ACC016',35.5,'DEBIT','2023-07-18','DINING','Pasta restaurant'),
    ('TXN0127','ACC016',480.0,'DEBIT','2023-08-05','TRAVEL','Budget airline tickets'),
    ('TXN0128','ACC016',55.75,'DEBIT','2023-08-22','GROCERIES','Weekly top-up'),
    ('TXN0129','ACC017',2200.0,'CREDIT','2023-06-15','RETAIL','Salary payment'),
    ('TXN0130','ACC017',112.0,'DEBIT','2023-07-02','GROCERIES','Monthly groceries'),
    ('TXN0131','ACC017',395.0,'DEBIT','2023-07-20','RETAIL','Home improvement'),
    ('TXN0132','ACC017',58.25,'DEBIT','2023-08-07','DINING','Italian restaurant'),
    ('TXN0133','ACC017',145.0,'DEBIT','2023-08-24','HEALTHCARE','Therapy session'),
    ('TXN0134','ACC018',2850.0,'DEBIT','2023-06-17','TRAVEL','Holiday resort'),
    ('TXN0135','ACC018',165.75,'DEBIT','2023-07-04','RETAIL','Summer fashion'),
    ('TXN0136','ACC018',78.5,'DEBIT','2023-07-22','DINING','Asian cuisine dinner'),
    ('TXN0137','ACC018',340.0,'DEBIT','2023-08-09','ENTERTAINMENT','Sports event tickets'),
    ('TXN0138','ACC018',92.25,'DEBIT','2023-08-26','HEALTHCARE','Dermatology visit'),
    ('TXN0139','ACC019',63.5,'DEBIT','2023-06-19','GROCERIES','Local market'),
    ('TXN0140','ACC019',185.0,'DEBIT','2023-07-06','UTILITIES','Phone and internet'),
    ('TXN0141','ACC019',48.75,'DEBIT','2023-07-24','DINING','Pizza night'),
    ('TXN0142','ACC019',275.0,'DEBIT','2023-08-11','RETAIL','Electronics accessories'),
    ('TXN0143','ACC019',700.0,'TRANSFER','2023-08-28','RETAIL','Monthly rent split'),
    ('TXN0144','ACC020',1600.0,'CREDIT','2023-06-21','RETAIL','Part-time income'),
    ('TXN0145','ACC020',86.25,'DEBIT','2023-07-08','GROCERIES','Organic groceries'),
    ('TXN0146','ACC020',52.5,'DEBIT','2023-07-26','DINING','Brunch cafe'),
    ('TXN0147','ACC020',380.0,'DEBIT','2023-08-13','HEALTHCARE','Dental checkup'),
    ('TXN0148','ACC020',125.0,'DEBIT','2023-08-30','ENTERTAINMENT','Theater tickets'),
    ('TXN0149','ACC021',1900.0,'DEBIT','2023-09-05','TRAVEL','International flight tickets'),
    ('TXN0150','ACC021',245.0,'DEBIT','2023-09-22','RETAIL','Books and electronics'),
    ('TXN0151','ACC021',88.5,'DEBIT','2023-10-10','DINING','Spanish tapas'),
    ('TXN0152','ACC021',3350.0,'DEBIT','2023-10-28','TRAVEL','All-inclusive resort'),
    ('TXN0153','ACC022',72.25,'DEBIT','2023-09-07','GROCERIES','Local grocery'),
    ('TXN0154','ACC022',160.0,'DEBIT','2023-09-24','UTILITIES','Water and electric'),
    ('TXN0155','ACC022',38.5,'DEBIT','2023-10-12','DINING','Cafe lunch'),
    ('TXN0156','ACC022',290.0,'DEBIT','2023-10-30','RETAIL','Winter clothing'),
    ('TXN0157','ACC023',3500.0,'CREDIT','2023-09-09','RETAIL','Freelance project'),
    ('TXN0158','ACC023',125.75,'DEBIT','2023-09-26','GROCERIES','Grocery delivery'),
    ('TXN0159','ACC023',450.0,'DEBIT','2023-10-14','HEALTHCARE','Full health checkup'),
    ('TXN0160','ACC023',67.5,'DEBIT','2023-10-31','DINING','Sushi bar'),
    ('TXN0161','ACC024',4500.0,'DEBIT','2023-09-11','TRAVEL','Europe trip package'),
    ('TXN0162','ACC024',185.25,'DEBIT','2023-09-28','DINING','Anniversary restaurant'),
    ('TXN0163','ACC024',920.0,'DEBIT','2023-10-16','RETAIL','Designer handbag'),
    ('TXN0164','ACC024',55.0,'DEBIT','2023-11-02','GROCERIES','Weekend groceries'),
    ('TXN0165','ACC024',2800.0,'DEBIT','2023-11-18','TRAVEL','Holiday booking'),
    ('TXN0166','ACC025',82.5,'DEBIT','2023-09-13','GROCERIES','Bulk groceries'),
    ('TXN0167','ACC025',235.0,'DEBIT','2023-09-30','UTILITIES','Combined utilities'),
    ('TXN0168','ACC025',58.75,'DEBIT','2023-10-18','DINING','Mexican restaurant'),
    ('TXN0169','ACC025',475.0,'DEBIT','2023-11-05','HEALTHCARE','Hospital consultation'),
    ('TXN0170','ACC026',1400.0,'CREDIT','2023-09-15','RETAIL','Monthly salary'),
    ('TXN0171','ACC026',95.25,'DEBIT','2023-10-02','GROCERIES','Supermarket shop'),
    ('TXN0172','ACC026',42.75,'DEBIT','2023-10-20','DINING','Indian cuisine'),
    ('TXN0173','ACC026',310.0,'DEBIT','2023-11-07','ENTERTAINMENT','Annual gym membership'),
    ('TXN0174','ACC027',5000.0,'DEBIT','2023-09-17','TRAVEL','Business trip Asia'),
    ('TXN0175','ACC027',320.5,'DEBIT','2023-10-04','RETAIL','Premium clothing'),
    ('TXN0176','ACC027',145.0,'DEBIT','2023-10-22','DINING','Executive club dinner'),
    ('TXN0177','ACC027',3800.0,'DEBIT','2023-11-09','TRAVEL','Family vacation package'),
    ('TXN0178','ACC027',278.0,'DEBIT','2023-11-25','HEALTHCARE','Executive health screen'),
    ('TXN0179','ACC028',62.75,'DEBIT','2023-09-19','GROCERIES','Grocery run'),
    ('TXN0180','ACC028',198.0,'DEBIT','2023-10-06','UTILITIES','Rent utilities share'),
    ('TXN0181','ACC028',45.5,'DEBIT','2023-10-24','DINING','Burger place'),
    ('TXN0182','ACC028',155.0,'DEBIT','2023-11-11','RETAIL','Gift shopping'),
    ('TXN0183','ACC029',2000.0,'CREDIT','2023-09-21','RETAIL','Bonus payment'),
    ('TXN0184','ACC029',118.5,'DEBIT','2023-10-08','GROCERIES','Family groceries'),
    ('TXN0185','ACC029',68.25,'DEBIT','2023-10-26','DINING','Chinese restaurant'),
    ('TXN0186','ACC029',880.0,'DEBIT','2023-11-13','TRAVEL','Thanksgiving travel'),
    ('TXN0187','ACC030',2300.0,'DEBIT','2023-09-23','TRAVEL','Mexico resort'),
    ('TXN0188','ACC030',158.75,'DEBIT','2023-10-10','RETAIL','Cosmetics and skincare'),
    ('TXN0189','ACC030',72.0,'DEBIT','2023-10-28','DINING','Steakhouse dinner'),
    ('TXN0190','ACC030',430.0,'DEBIT','2023-11-14','HEALTHCARE','Physiotherapy sessions'),
    ('TXN0191','ACC031',78.25,'DEBIT','2023-09-25','GROCERIES','Local supermarket'),
    ('TXN0192','ACC031',145.0,'DEBIT','2023-10-12','UTILITIES','Internet provider'),
    ('TXN0193','ACC031',360.0,'DEBIT','2023-10-30','RETAIL','Tool kit purchase'),
    ('TXN0194','ACC031',55.5,'DEBIT','2023-11-16','DINING','Diner visit'),
    ('TXN0195','ACC032',1750.0,'CREDIT','2023-09-27','RETAIL','Project income'),
    ('TXN0196','ACC032',88.0,'DEBIT','2023-10-14','GROCERIES','Health food store'),
    ('TXN0197','ACC032',295.0,'DEBIT','2023-11-01','HEALTHCARE','Yoga and wellness'),
    ('TXN0198','ACC032',42.5,'DEBIT','2023-11-18','DINING','Smoothie bar'),
    ('TXN0199','ACC033',1500.0,'DEBIT','2023-09-29','TRAVEL','Skiing trip deposit'),
    ('TXN0200','ACC033',425.75,'DEBIT','2023-10-16','RETAIL','Winter sports gear'),
    ('TXN0201','ACC033',92.5,'DEBIT','2023-11-03','DINING','Sports bar'),
    ('TXN0202','ACC033',3100.0,'DEBIT','2023-11-20','TRAVEL','Christmas travel package'),
    ('TXN0203','ACC034',65.5,'DEBIT','2023-10-01','GROCERIES','Grocery delivery'),
    ('TXN0204','ACC034',175.0,'DEBIT','2023-10-19','UTILITIES','Household bills'),
    ('TXN0205','ACC034',38.25,'DEBIT','2023-11-06','DINING','Fast food meal'),
    ('TXN0206','ACC034',220.0,'DEBIT','2023-11-22','RETAIL','Black Friday deals'),
    ('TXN0207','ACC035',3200.0,'CREDIT','2023-10-03','RETAIL','Quarterly salary'),
    ('TXN0208','ACC035',135.75,'DEBIT','2023-10-21','GROCERIES','Bulk grocery order'),
    ('TXN0209','ACC035',58.0,'DEBIT','2023-11-08','DINING','Japanese restaurant'),
    ('TXN0210','ACC035',790.0,'DEBIT','2023-11-24','ENTERTAINMENT','Holiday show tickets'),
    ('TXN0211','ACC001',55.25,'DEBIT','2024-01-08','GROCERIES','New year groceries'),
    ('TXN0212','ACC001',980.0,'DEBIT','2024-01-25','TRAVEL','Winter break travel'),
    ('TXN0213','ACC001',42.5,'DEBIT','2024-02-12','DINING','Valentine day dinner'),
    ('TXN0214','ACC001',115.0,'DEBIT','2024-02-29','UTILITIES','February bills'),
    ('TXN0215','ACC002',2800.0,'CREDIT','2024-01-10','RETAIL','Monthly salary'),
    ('TXN0216','ACC002',88.75,'DEBIT','2024-01-27','GROCERIES','January groceries'),
    ('TXN0217','ACC002',165.0,'DEBIT','2024-02-14','HEALTHCARE','Annual checkup'),
    ('TXN0218','ACC002',310.0,'DEBIT','2024-03-03','RETAIL','Spring wardrobe'),
    ('TXN0219','ACC003',3600.0,'DEBIT','2024-01-12','TRAVEL','CES conference trip'),
    ('TXN0220','ACC003',175.5,'DEBIT','2024-02-01','DINING','Business dinner'),
    ('TXN0221','ACC003',540.0,'DEBIT','2024-02-18','RETAIL','Office equipment'),
    ('TXN0222','ACC003',89.25,'DEBIT','2024-03-07','GROCERIES','Home delivery'),
    ('TXN0223','ACC004',78.0,'DEBIT','2024-01-14','GROCERIES','January shop'),
    ('TXN0224','ACC004',225.0,'DEBIT','2024-01-31','UTILITIES','January utilities'),
    ('TXN0225','ACC004',48.5,'DEBIT','2024-02-16','DINING','Cafe breakfast'),
    ('TXN0226','ACC005',1200.0,'CREDIT','2024-01-16','RETAIL','New contract payment'),
    ('TXN0227','ACC005',92.25,'DEBIT','2024-02-03','GROCERIES','Weekly groceries'),
    ('TXN0228','ACC005',385.0,'DEBIT','2024-02-20','HEALTHCARE','Specialist appointment'),
    ('TXN0229','ACC006',2750.0,'DEBIT','2024-01-18','TRAVEL','Q1 business travel'),
    ('TXN0230','ACC006',195.75,'DEBIT','2024-02-05','RETAIL','Professional wardrobe'),
    ('TXN0231','ACC006',68.0,'DEBIT','2024-02-22','DINING','Power lunch'),
    ('TXN0232','ACC007',62.5,'DEBIT','2024-01-20','GROCERIES','Grocery run'),
    ('TXN0233','ACC007',280.0,'DEBIT','2024-02-07','UTILITIES','February bills'),
    ('TXN0234','ACC007',745.0,'DEBIT','2024-03-01','TRAVEL','Spring break trip'),
    ('TXN0235','ACC008',3200.0,'CREDIT','2024-01-22','RETAIL','January salary'),
    ('TXN0236','ACC008',108.5,'DEBIT','2024-02-09','GROCERIES','February groceries'),
    ('TXN0237','ACC008',52.75,'DEBIT','2024-02-26','DINING','Sunday brunch'),
    ('TXN0238','ACC009',4200.0,'DEBIT','2024-01-24','TRAVEL','Winter escape resort'),
    ('TXN0239','ACC009',385.0,'DEBIT','2024-02-11','RETAIL','Luxury goods'),
    ('TXN0240','ACC009',245.5,'DEBIT','2024-02-28','DINING','Michelin star dinner'),
    ('TXN0241','ACC010',68.25,'DEBIT','2024-01-26','GROCERIES','January top-up'),
    ('TXN0242','ACC010',195.0,'DEBIT','2024-02-13','UTILITIES','Winter heating'),
    ('TXN0243','ACC010',560.0,'DEBIT','2024-03-02','RETAIL','Tax preparation services'),
    ('TXN0244','ACC011',95.75,'DEBIT','2024-03-10','GROCERIES','Grocery shop'),
    ('TXN0245','ACC011',420.0,'DEBIT','2024-03-27','TRAVEL','Easter trip'),
    ('TXN0246','ACC011',62.25,'DEBIT','2024-04-14','DINING','Family dinner'),
    ('TXN0247','ACC012',1800.0,'DEBIT','2024-03-12','RETAIL','Spring electronics'),
    ('TXN0248','ACC012',78.5,'DEBIT','2024-03-29','GROCERIES','Organic shop'),
    ('TXN0249','ACC012',550.0,'DEBIT','2024-04-16','TRAVEL','Long weekend trip'),
    ('TXN0250','ACC013',55.0,'DEBIT','2024-03-14','GROCERIES','Local market'),
    ('TXN0251','ACC013',195.0,'DEBIT','2024-04-01','UTILITIES','Spring bills'),
    ('TXN0252','ACC013',42.25,'DEBIT','2024-04-18','DINING','Fast casual lunch'),
    ('TXN0253','ACC014',2100.0,'CREDIT','2024-03-16','RETAIL','Q1 freelance income'),
    ('TXN0254','ACC014',115.25,'DEBIT','2024-04-03','GROCERIES','Easter groceries'),
    ('TXN0255','ACC014',380.0,'DEBIT','2024-04-20','HEALTHCARE','Follow-up procedures'),
    ('TXN0256','ACC015',3800.0,'DEBIT','2024-03-18','TRAVEL','Spring conference'),
    ('TXN0257','ACC015',225.0,'DEBIT','2024-04-05','RETAIL','Business attire'),
    ('TXN0258','ACC016',72.5,'DEBIT','2024-03-20','GROCERIES','Grocery trip'),
    ('TXN0259','ACC016',310.0,'DEBIT','2024-04-07','UTILITIES','Spring cleaning bills'),
    ('TXN0260','ACC016',58.75,'DEBIT','2024-04-24','DINING','Neighborhood bistro'),
    ('TXN0261','ACC017',2500.0,'CREDIT','2024-03-22','RETAIL','Q1 salary'),
    ('TXN0262','ACC017',125.0,'DEBIT','2024-04-09','GROCERIES','Monthly shop'),
    ('TXN0263','ACC017',495.0,'DEBIT','2024-04-26','TRAVEL','Weekend city break'),
    ('TXN0264','ACC018',3100.0,'DEBIT','2024-03-24','TRAVEL','Spring vacation'),
    ('TXN0265','ACC018',178.25,'DEBIT','2024-04-11','RETAIL','Spring fashion'),
    ('TXN0266','ACC018',85.5,'DEBIT','2024-04-28','DINING','Fusion restaurant'),
    ('TXN0267','ACC019',65.75,'DEBIT','2024-03-26','GROCERIES','Grocery delivery'),
    ('TXN0268','ACC019',215.0,'DEBIT','2024-04-13','UTILITIES','April bills'),
    ('TXN0269','ACC019',44.5,'DEBIT','2024-04-30','DINING','Vietnamese takeout'),
    ('TXN0270','ACC020',1800.0,'CREDIT','2024-03-28','RETAIL','Spring income'),
    ('TXN0271','ACC020',98.5,'DEBIT','2024-04-15','GROCERIES','Organic market shop'),
    ('TXN0272','ACC020',62.25,'DEBIT','2024-05-02','DINING','Brunch spot'),
    ('TXN0273','ACC020',445.0,'DEBIT','2024-05-19','HEALTHCARE','Quarterly wellness visit'),
    ('TXN0274','ACC001',68.5,'DEBIT','2024-03-15','GROCERIES','Spring groceries'),
    ('TXN0275','ACC002',52.75,'DEBIT','2024-03-20','DINING','Team lunch'),
    ('TXN0276','ACC003',145.0,'DEBIT','2024-04-02','HEALTHCARE','Executive health check'),
    ('TXN0277','ACC004',320.0,'DEBIT','2024-03-18','RETAIL','Spring wardrobe'),
    ('TXN0278','ACC005',76.5,'DEBIT','2024-03-12','GROCERIES','Local market'),
    ('TXN0279','ACC006',490.0,'DEBIT','2024-03-25','HEALTHCARE','Specialist consultation'),
    ('TXN0280','ACC007',88.25,'DEBIT','2024-04-05','GROCERIES','April groceries'),
    ('TXN0281','ACC008',165.0,'DEBIT','2024-04-10','RETAIL','Spring sale'),
    ('TXN0282','ACC009',1950.0,'DEBIT','2024-04-02','ENTERTAINMENT','Charity gala tickets'),
    ('TXN0283','ACC010',42.75,'DEBIT','2024-04-08','DINING','Lunch break'),
    ('TXN0284','ACC021',78.0,'DEBIT','2024-04-05','GROCERIES','Grocery run'),
    ('TXN0285','ACC022',185.0,'DEBIT','2024-04-15','UTILITIES','April utilities'),
    ('TXN0286','ACC023',55.25,'DEBIT','2024-04-20','DINING','Pizza night out'),
    ('TXN0287','ACC024',1650.0,'DEBIT','2024-05-01','TRAVEL','Summer booking'),
    ('TXN0288','ACC025',92.5,'DEBIT','2024-04-25','GROCERIES','Grocery shop'),
    ('TXN0289','ACC026',275.0,'DEBIT','2024-05-05','HEALTHCARE','Dental work'),
    ('TXN0290','ACC027',4100.0,'DEBIT','2024-04-28','TRAVEL','Premium vacation'),
    ('TXN0291','ACC028',58.25,'DEBIT','2024-05-08','GROCERIES','Weekly groceries'),
    ('TXN0292','ACC029',345.0,'DEBIT','2024-05-12','RETAIL','Home electronics'),
    ('TXN0293','ACC030',82.75,'DEBIT','2024-05-15','DINING','Spanish restaurant'),
    ('TXN0294','ACC031',195.0,'DEBIT','2024-05-18','UTILITIES','May bills'),
    ('TXN0295','ACC032',2400.0,'CREDIT','2024-05-01','RETAIL','Q2 project income'),
    ('TXN0296','ACC033',145.5,'DEBIT','2024-05-22','RETAIL','Outdoor gear'),
    ('TXN0297','ACC034',65.75,'DEBIT','2024-05-25','GROCERIES','Monthly groceries'),
    ('TXN0298','ACC035',880.0,'DEBIT','2024-06-01','TRAVEL','Summer getaway booking'),
    ('TXN0299','ACC036',48.25,'DEBIT','2023-03-15','GROCERIES','Inactive account purchase'),
    ('TXN0300','ACC046',2200.0,'DEBIT','2023-06-22','TRAVEL','Frozen account large purchase'),
]

transactions_df = session.create_dataframe(transactions_data, schema=transactions_schema)
transactions_df.write.mode('overwrite').save_as_table('TRANSACTIONS')
row_count = session.table('TRANSACTIONS').count()
print(f'TRANSACTIONS loaded: {row_count} rows')
session.table('TRANSACTIONS').show(5)

---

## Section 3: The PySpark Source Code

The file `transaction_pipeline.py` is the PySpark pipeline we will migrate. Below are the key patterns the SMA tool converts, annotated with what changes.

### Pattern 1: Session Initialization

```python
# BEFORE — PySpark (does not run natively in Snowflake)
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('FinancialTransactionPipeline') \
    .getOrCreate()
```

**What changes:** `SparkSession.builder` is replaced by Snowpark's `get_active_session()`. The `.appName()` call and any connector JAR configuration are removed entirely.

### Pattern 2: Reading via the Spark-Snowflake Connector

```python
# BEFORE — PySpark + Snowflake connector
accounts_df = spark.read \
    .format('net.snowflake.spark.snowflake') \
    .options(**SF_OPTIONS) \
    .option('dbtable', 'ACCOUNTS') \
    .load()
```

**What changes:** The connector format and connection options dictionary are replaced by `session.table('ACCOUNTS')`. Since Snowpark runs *inside* Snowflake, no external connector is needed.

### Pattern 3: User-Defined Functions (UDFs)

```python
# BEFORE — PySpark UDF
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

categorize_amount_udf = F.udf(categorize_amount, StringType())
df.withColumn('amount_category', categorize_amount_udf(F.col('amount')))
```

**What changes:**
- `F.udf(func, ReturnType)` becomes `udf(func, return_type=StringType(), input_types=[DoubleType()])`
- `.withColumn()` becomes `.with_column()` (snake_case)
- Import: `from snowflake.snowpark.functions import udf`

### Pattern 4: Window Functions

```python
# BEFORE — PySpark window
from pyspark.sql.window import Window

w = Window.partitionBy('account_id').orderBy('year_month')
df.withColumn('running_total', F.sum('amount').over(w))
```

**What changes:** `partitionBy` → `partition_by`, `orderBy` → `order_by`, `unboundedPreceding` → `UNBOUNDED_PRECEDING`, `currentRow` → `CURRENT_ROW`.

### Pattern 5: Writing Back to Snowflake

```python
# BEFORE — connector write
result_df.write \
    .format('net.snowflake.spark.snowflake') \
    .options(**SF_OPTIONS) \
    .option('dbtable', 'MONTHLY_SPEND_SUMMARY') \
    .mode('overwrite').save()
```

**What changes:** Replaced by `result_df.write.mode('overwrite').save_as_table('MONTHLY_SPEND_SUMMARY')` — far simpler because the DataFrame is already running inside Snowflake.

> Open `transaction_pipeline.py` in your editor to see the full source before running the migration skill.

---

## Section 4: Run the Migration Skill in Cortex Code

This section runs **outside this notebook** on your local machine using Cortex Code.

### Step 1: Trigger the Skill

Open Cortex Code and type any of the following:
```
migrate pyspark
convert spark to snowflake
run sma
sma conversion
```
The `snowflake-migration` skill activates automatically.

### Step 2: Provide Project Information

| Field | Value for this lab |
|-------|-------------------|
| Original Source Code Path | Full path to `test_workload_for_demo/` |
| Customer Email | your email address |
| Customer Company | your company name |
| Project Name | `finance-pipeline-lab` |

### Step 3: Choose Migration Status

Select **(b) I want to run the conversion now**.

### Step 4: Output Path

Enter a new empty folder, for example `~/sma-output/finance-lab/`.

### Step 5: Conversion Tool

Select **(c) Snowpark API**.

### Step 6: Configure SMA CLI (First Run Only)

Provide the path to your SMA binary, e.g. `/path/to/SMA-CLI/orchestrator/sma`.

Confirm these options:

| Option | Setting |
|--------|---------|
| Jupyter Conversion | Y |
| SQL Flavor | SparkSql |
| Generate Checkpoints | Y |

### Step 7: Confirm and Run

Type **yes** to start the conversion. Progress is printed as each SMA step completes:
```
[SMA] Step 1/20 - FileDiscovery: STARTED
[SMA] Step 4/20 - PythonConversion: STARTED
...
Conversion was successful.
```

### Step 8: Review the EWI Dashboard

Once conversion completes, the skill opens the EWI dashboard in your browser automatically.

> Continue to **Section 5** to understand what the dashboard shows, then return here.

---

## Section 5: Understanding the EWI Dashboard

The dashboard shows every flagged issue in the converted code.

### What is an EWI?

**EWI** = **Error, Warning, or Issue** — a marker left where the SMA tool could not perform a fully automatic conversion.

| Severity | Meaning |
|----------|---------|
| **Error** | Code will not run without a manual fix |
| **Warning** | Code may run but behavior could differ from original |
| **Info** | Informational note — usually safe to ignore |

### EWIs to Expect from This Workload

| PySpark pattern | Why flagged | Severity |
|-----------------|-------------|----------|
| `net.snowflake.spark.snowflake` read/write | Connector replaced by native Snowpark API | Warning |
| `SparkSession.builder` | Replaced by `get_active_session()` | Info |
| `F.udf(func, ReturnType)` | UDF registration syntax differs in Snowpark | Warning |
| `Window.partitionBy()` | Renamed `partition_by()` in Snowpark | Info |
| `.withColumn()` | Renamed `.with_column()` in Snowpark | Info |

### Running the EWI Fixer

When the skill asks *'Would you like to run the EWI Fixer?'*, select **Yes** with these settings:

- **EWI comment handling:** Mark (keeps comments prefixed with [FIXED] or [NOT-FIXED])
- **Which EWIs to process:** Only pending (first run)

The EWI Fixer scans all converted files and applies automatic fixes. The dashboard updates with resolved counts when it finishes.

> After the EWI Fixer completes, **return to this notebook** and run Section 6.

---

## Section 6: The Converted Snowpark Pipeline

The cells below implement the full Snowpark equivalent of `transaction_pipeline.py`. Run them in order.

### PySpark → Snowpark API Translation

| PySpark | Snowpark | Notes |
|---------|----------|-------|
| `SparkSession.builder.getOrCreate()` | `get_active_session()` | No setup needed inside Snowflake |
| `from pyspark.sql import functions as F` | `from snowflake.snowpark import functions as F` | Same alias, different import |
| `from pyspark.sql.window import Window` | `from snowflake.snowpark.window import Window` | Same class, different import |
| `F.udf(func, ReturnType)` | `udf(func, return_type=..., input_types=[...])` | Explicit input types required |
| `df.withColumn(name, col)` | `df.with_column(name, col)` | Snake_case |
| `df.groupBy(...)` | `df.group_by(...)` | Snake_case |
| `Window.partitionBy(...)` | `Window.partition_by(...)` | Snake_case |
| `Window.orderBy(...)` | `Window.order_by(...)` | Snake_case |
| `Window.unboundedPreceding` | `Window.UNBOUNDED_PRECEDING` | UPPER_CASE constant |
| `Window.currentRow` | `Window.CURRENT_ROW` | UPPER_CASE constant |
| `col.isNotNull()` | `col.is_not_null()` | Snake_case |
| `F.date_format(col, 'yyyy-MM')` | `F.to_char(col, 'YYYY-MM')` | Snowflake date format string |
| `connector.write(...).save()` | `df.write.save_as_table('TABLE')` | Native Snowpark write |
| `spark.read.format(connector).load()` | `session.table('TABLE')` | Native session read |

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.window import Window
from snowflake.snowpark.types import StringType, DoubleType
from snowflake.snowpark.functions import udf

session = get_active_session()
print('Session ready.')

# ── UDF: categorise transaction amount into size bands ─────────────────────────
# PySpark: F.udf(func, StringType())
# Snowpark: udf(func, return_type=StringType(), input_types=[DoubleType()])
def _categorize_amount(amount):
    if amount is None:
        return 'unknown'
    if amount < 50.0:
        return 'small'
    elif amount < 500.0:
        return 'medium'
    else:
        return 'large'

categorize_amount_udf = udf(_categorize_amount, return_type=StringType(), input_types=[DoubleType()])

# ── UDF: simple fraud-flag heuristic ──────────────────────────────────────────
def _fraud_flag(account_type, amount):
    if account_type == 'CREDIT' and amount is not None and amount > 3000.0:
        return 'REVIEW'
    return 'OK'

fraud_flag_udf = udf(_fraud_flag, return_type=StringType(), input_types=[StringType(), DoubleType()])

print('UDFs registered.')

In [ ]:
# ── Step 1: Read source tables ─────────────────────────────────────────────────
# PySpark: spark.read.format('net.snowflake.spark.snowflake').option('dbtable', ...).load()
# Snowpark: session.table('TABLE_NAME')  — no connector needed

accounts_df     = session.table('ACCOUNTS')
transactions_df = session.table('TRANSACTIONS')

print(f'ACCOUNTS rows    : {accounts_df.count()}')
print(f'TRANSACTIONS rows: {transactions_df.count()}')

In [ ]:
# ── Step 2: Filter & Select ────────────────────────────────────────────────────
# .filter() and .select() work identically in Snowpark
# .isNotNull()  ->  .is_not_null()  (snake_case)

active_accounts = (
    accounts_df
    .filter(F.col('STATUS') == 'ACTIVE')
    .select(
        'ACCOUNT_ID',
        'CUSTOMER_NAME',
        'ACCOUNT_TYPE',
        F.col('CREDIT_LIMIT').cast('double').alias('CREDIT_LIMIT'),
    )
)

filtered_txns = (
    transactions_df
    .filter(F.col('TRANSACTION_TYPE').isin('DEBIT', 'TRANSFER'))
    .filter(F.col('AMOUNT').is_not_null())
    .select(
        'TRANSACTION_ID',
        'ACCOUNT_ID',
        F.col('AMOUNT').cast('double').alias('AMOUNT'),
        'TRANSACTION_TYPE',
        F.to_date('TRANSACTION_DATE', 'YYYY-MM-DD').alias('TRANSACTION_DATE'),
        'MERCHANT_CATEGORY',
        'DESCRIPTION',
    )
)

print(f'Active accounts      : {active_accounts.count()}')
print(f'Filtered transactions: {filtered_txns.count()}')

In [ ]:
# ── Step 3: Apply UDFs ────────────────────────────────────────────────────────
# PySpark: .withColumn()  ->  Snowpark: .with_column()  (snake_case)

enriched_txns = filtered_txns.with_column(
    'AMOUNT_CATEGORY',
    categorize_amount_udf(F.col('AMOUNT'))
)

# ── Step 4: Join ───────────────────────────────────────────────────────────────
joined_df = enriched_txns.join(active_accounts, on='ACCOUNT_ID', how='inner')

joined_df = joined_df.with_column(
    'FRAUD_FLAG',
    fraud_flag_udf(F.col('ACCOUNT_TYPE'), F.col('AMOUNT'))
)

print(f'Joined rows: {joined_df.count()}')
joined_df.select('ACCOUNT_ID', 'AMOUNT', 'AMOUNT_CATEGORY', 'FRAUD_FLAG').show(5)

In [ ]:
# ── Step 5: Aggregation ────────────────────────────────────────────────────────
# PySpark: .groupBy()    ->  Snowpark: .group_by()   (snake_case)
# F.date_format(col, 'yyyy-MM')  ->  F.to_char(col, 'YYYY-MM')
# All agg functions (sum, count, avg, max, when) are identical

monthly_spend = (
    joined_df
    .with_column('YEAR_MONTH', F.to_char(F.col('TRANSACTION_DATE'), 'YYYY-MM'))
    .group_by(
        'ACCOUNT_ID', 'CUSTOMER_NAME', 'ACCOUNT_TYPE',
        'CREDIT_LIMIT', 'YEAR_MONTH', 'MERCHANT_CATEGORY',
    )
    .agg(
        F.sum('AMOUNT').alias('TOTAL_SPEND'),
        F.count('TRANSACTION_ID').alias('TRANSACTION_COUNT'),
        F.avg('AMOUNT').alias('AVG_TRANSACTION_AMOUNT'),
        F.max('AMOUNT').alias('MAX_TRANSACTION_AMOUNT'),
        F.sum(
            F.when(F.col('FRAUD_FLAG') == 'REVIEW', F.lit(1)).otherwise(F.lit(0))
        ).alias('FLAGGED_TRANSACTIONS'),
    )
)

print(f'Monthly aggregation rows: {monthly_spend.count()}')
monthly_spend.show(5)

In [ ]:
# ── Step 6: Window Functions ───────────────────────────────────────────────────
# PySpark                          Snowpark
# Window.partitionBy(...)    ->    Window.partition_by(...)
# Window.orderBy(...)        ->    Window.order_by(...)
# Window.unboundedPreceding  ->    Window.UNBOUNDED_PRECEDING
# Window.currentRow          ->    Window.CURRENT_ROW

account_month_window = (
    Window.partition_by('ACCOUNT_ID')
    .order_by('YEAR_MONTH')
    .rows_between(Window.UNBOUNDED_PRECEDING, Window.CURRENT_ROW)
)

global_rank_window = Window.order_by(F.col('TOTAL_SPEND').desc())

lag_window = (
    Window.partition_by('ACCOUNT_ID', 'MERCHANT_CATEGORY')
    .order_by('YEAR_MONTH')
)

result_df = (
    monthly_spend
    .with_column('CUMULATIVE_SPEND',        F.sum('TOTAL_SPEND').over(account_month_window))
    .with_column('SPEND_RANK',              F.rank().over(global_rank_window))
    .with_column('PREV_MONTH_SPEND',        F.lag('TOTAL_SPEND', 1).over(lag_window))
    .with_column('MONTH_OVER_MONTH_CHANGE', F.col('TOTAL_SPEND') - F.col('PREV_MONTH_SPEND'))
    .with_column(
        'CREDIT_UTILISATION_PCT',
        F.round((F.col('CUMULATIVE_SPEND') / F.col('CREDIT_LIMIT')) * 100, 2),
    )
)

# ── Step 7: Write to Snowflake ─────────────────────────────────────────────────
# PySpark: df.write.format(connector).option('dbtable', ...).mode('overwrite').save()
# Snowpark: df.write.mode('overwrite').save_as_table('TABLE_NAME')

result_df.write.mode('overwrite').save_as_table('MONTHLY_SPEND_SUMMARY')

row_count = session.table('MONTHLY_SPEND_SUMMARY').count()
print(f'Pipeline complete. {row_count} rows written to MONTHLY_SPEND_SUMMARY.')

---

## Section 7: Validate the Results

Run the queries below to confirm the pipeline produced the expected output.

In [ ]:
-- Row count and column check
SELECT
    COUNT(*)                                    AS total_rows,
    COUNT(DISTINCT account_id)                  AS distinct_accounts,
    COUNT(DISTINCT year_month)                  AS distinct_months,
    COUNT(DISTINCT merchant_category)           AS distinct_categories,
    SUM(flagged_transactions)                   AS total_flagged,
    ROUND(AVG(credit_utilisation_pct), 2)       AS avg_credit_utilisation_pct
FROM MONTHLY_SPEND_SUMMARY;

In [ ]:
-- Top 10 accounts by total spend (all months combined)
SELECT
    account_id,
    customer_name,
    account_type,
    ROUND(SUM(total_spend), 2)   AS lifetime_spend,
    SUM(transaction_count)       AS total_transactions,
    MAX(credit_utilisation_pct)  AS peak_credit_utilisation_pct
FROM MONTHLY_SPEND_SUMMARY
GROUP BY account_id, customer_name, account_type
ORDER BY lifetime_spend DESC
LIMIT 10;

In [ ]:
-- Fraud-flagged rows: CREDIT accounts with high-value transactions
SELECT
    account_id,
    customer_name,
    year_month,
    merchant_category,
    total_spend,
    flagged_transactions
FROM MONTHLY_SPEND_SUMMARY
WHERE flagged_transactions > 0
ORDER BY total_spend DESC
LIMIT 15;

In [ ]:
# Preview the full output with all window function columns
session.table('MONTHLY_SPEND_SUMMARY') \
    .select(
        'ACCOUNT_ID', 'CUSTOMER_NAME', 'YEAR_MONTH', 'MERCHANT_CATEGORY',
        'TOTAL_SPEND', 'CUMULATIVE_SPEND', 'SPEND_RANK',
        'PREV_MONTH_SPEND', 'MONTH_OVER_MONTH_CHANGE', 'CREDIT_UTILISATION_PCT'
    ) \
    .sort('SPEND_RANK') \
    .show(10)

---

## Section 8: Cleanup (Optional)

Uncomment and run the cell below to remove all objects created by this lab.

In [ ]:
-- Uncomment to clean up all lab objects
-- DROP DATABASE IF EXISTS DEMO_DB;
-- DROP WAREHOUSE IF EXISTS LAB_WH;